# Tests for multirun.py code fixing functionality

Testing the helper functions for code fixing in multirun.py.


In [32]:
# Reload modules after code changes (run this cell after editing multirun.py or fix_code.py)
import importlib
import sys

# Reload the modules containing our functions
if 'stat_genie.blade_pipeline.baselines.multirun' in sys.modules:
    importlib.reload(sys.modules['stat_genie.blade_pipeline.baselines.multirun'])
if 'stat_genie.blade_pipeline.additions.analysis.fix_code' in sys.modules:
    importlib.reload(sys.modules['stat_genie.blade_pipeline.additions.analysis.fix_code'])

print("Modules reloaded")


Modules reloaded


## Setup and Imports


In [33]:
import os
import tempfile
import json
from pathlib import Path

from blade_bench.eval.datamodel.lm_analysis import (
    AgentCVarsWithCol,
    ControlVarWithCol,
    DVarWithCol,
    EntireAnalysis,
    IVarWithCol,
)
from blade_bench.eval.utils import SAVE_CODE_TEMPLATE

# Import the functions we're testing
from stat_genie.blade_pipeline.baselines.multirun import (
    _extract_code_from_file,
    _update_analysis_with_fixed_code,
    _format_cvars_for_prompt,
)
from stat_genie.blade_pipeline.additions.analysis.fix_code import (
    check_and_fix_code_with_cvars,
)


## Test Fixtures


In [34]:
# Create sample cvars
sample_cvars = AgentCVarsWithCol(
    ivs=[
        IVarWithCol(
            description="Test independent variable",
            columns=["test_iv"]
        )
    ],
    dv=DVarWithCol(
        description="Test dependent variable",
        columns=["test_dv"]
    ),
    controls=[
        ControlVarWithCol(
            description="Test control variable",
            is_moderator=False,
            moderator_on=None,
            columns=["test_control"]
        )
    ]
)

# Create sample EntireAnalysis
sample_entire_analysis = EntireAnalysis(
    cvars=sample_cvars,
    transform_code="def transform(df: pd.DataFrame) -> pd.DataFrame:\n    df['test_iv'] = df['x']\n    return df",
    m_code="def model(df: pd.DataFrame) -> Any:\n    return df['test_iv'].mean()"
)

# Create sample .py file content
sample_py_file_content = SAVE_CODE_TEMPLATE.format(
    data_path="/path/to/data.csv",
    transform_code="def transform(df: pd.DataFrame) -> pd.DataFrame:\n    df['test_iv'] = df['x']\n    return df",
    model_code="def model(df: pd.DataFrame) -> Any:\n    return df['test_iv'].mean()"
)

print("Test fixtures created")


Test fixtures created


## Test 1: Extract code from file


In [35]:
with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(sample_py_file_content)
    temp_path = f.name

try:
    transform_code, model_code = _extract_code_from_file(temp_path)
    
    expected_transform = "def transform(df: pd.DataFrame) -> pd.DataFrame:\n    df['test_iv'] = df['x']\n    return df"
    expected_model = "def model(df: pd.DataFrame) -> Any:\n    return df['test_iv'].mean()"
    
    assert transform_code == expected_transform, f"Transform code mismatch"
    assert model_code == expected_model, f"Model code mismatch"
    
    print("Test 1 passed: Code extraction works correctly")
finally:
    os.unlink(temp_path)


Test 1 passed: Code extraction works correctly


## Test 2: Update Analysis with Fixed Code


In [36]:
original_cvars = sample_entire_analysis.cvars.model_dump()
original_transform = sample_entire_analysis.transform_code
original_model = sample_entire_analysis.m_code

new_transform_code = "def transform(df: pd.DataFrame) -> pd.DataFrame:\n    df['test_iv'] = df['x'] * 2\n    return df"
new_model_code = "def model(df: pd.DataFrame) -> Any:\n    return df['test_iv'].sum()"

updated_analysis = _update_analysis_with_fixed_code(
    sample_entire_analysis,
    new_transform_code,
    new_model_code
)

# Verify cvars are unchanged
assert updated_analysis.cvars.model_dump() == original_cvars, "cvars should be unchanged"

# Verify code is updated
assert updated_analysis.transform_code == new_transform_code, "transform_code should be updated"
assert updated_analysis.m_code == new_model_code, "m_code should be updated"

# Verify original object is unchanged (immutability)
assert sample_entire_analysis.transform_code == original_transform, "Original analysis should not be mutated"
assert sample_entire_analysis.m_code == original_model, "Original analysis should not be mutated"
assert sample_entire_analysis.cvars.model_dump() == original_cvars, "Original cvars should not be mutated"

print("Test 2 passed: Analysis updated while preserving cvars")


Test 2 passed: Analysis updated while preserving cvars


## Test 3: Format cvars for prompt


In [37]:
cvars_text = _format_cvars_for_prompt(sample_cvars)

# Check that all column names are present
assert "test_iv" in cvars_text, "IV column name should be in prompt"
assert "test_dv" in cvars_text, "DV column name should be in prompt"
assert "test_control" in cvars_text, "Control column name should be in prompt"

# Check that descriptions are present
assert "Test independent variable" in cvars_text, "IV description should be in prompt"
assert "Test dependent variable" in cvars_text, "DV description should be in prompt"

# Check format structure
assert "Independent variables:" in cvars_text, "Should have IV section header"
assert "Dependent variable:" in cvars_text, "Should have DV section header"
assert "Control variables:" in cvars_text, "Should have control section header"

print("Test 3 passed: cvars formatted correctly for prompt")
print("\nSample output:")
print(cvars_text[:200] + "...")


Test 3 passed: cvars formatted correctly for prompt

Sample output:
Independent variables:
  - Test independent variable : columns = ['test_iv']

Dependent variable:
  - Test dependent variable : columns = ['test_dv']

Control variables:
  - Test control variable : co...


## Test 4: Extract Code - error handling


In [38]:
# Test error handling for missing markers
with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write("def transform(df):\n    return df\n")
    temp_path = f.name

try:
    try:
        _extract_code_from_file(temp_path)
        assert False, "Should have raised ValueError for missing markers"
    except ValueError as e:
        assert "marker" in str(e).lower() or "transform" in str(e).lower(), f"Error message should mention markers: {e}"
        print("Test 4 passed: Error handling works for missing markers")
finally:
    os.unlink(temp_path)


Test 4 passed: Error handling works for missing markers


## Test 5: Format cvars with moderators


In [39]:
# Create cvars with a moderator
cvars_with_moderator = AgentCVarsWithCol(
    ivs=[
        IVarWithCol(
            description="Main independent variable",
            columns=["main_iv"]
        )
    ],
    dv=DVarWithCol(
        description="Dependent variable",
        columns=["dv"]
    ),
    controls=[
        ControlVarWithCol(
            description="Control variable",
            is_moderator=False,
            moderator_on=None,
            columns=["control"]
        ),
        ControlVarWithCol(
            description="Moderator variable",
            is_moderator=True,
            moderator_on="Main independent variable",
            columns=["moderator"]
        )
    ]
)

cvars_text = _format_cvars_for_prompt(cvars_with_moderator)

# Check moderator is mentioned
assert "Moderator on:" in cvars_text, "Should mention moderator relationship"
assert "Main independent variable" in cvars_text, "Should include what the moderator is on"

# Check all variables are present
assert "main_iv" in cvars_text
assert "dv" in cvars_text
assert "control" in cvars_text
assert "moderator" in cvars_text

print("Test 5 passed: cvars with moderators formatted correctly")
print("\nSample output:")
print(cvars_text)


Test 5 passed: cvars with moderators formatted correctly

Sample output:
Independent variables:
  - Main independent variable : columns = ['main_iv']

Dependent variable:
  - Dependent variable : columns = ['dv']

Control variables:
  - Control variable : columns = ['control']
  - Moderator variable : columns = ['moderator']
    (Moderator on: Main independent variable)


## Test 6: Multiple variables


In [40]:
# Create cvars with multiple IVs and controls
complex_cvars = AgentCVarsWithCol(
    ivs=[
        IVarWithCol(description="IV 1", columns=["iv1"]),
        IVarWithCol(description="IV 2", columns=["iv2"])
    ],
    dv=DVarWithCol(
        description="Dependent variable",
        columns=["dv"]
    ),
    controls=[
        ControlVarWithCol(
            description="Control 1",
            is_moderator=False,
            moderator_on=None,
            columns=["control1"]
        ),
        ControlVarWithCol(
            description="Control 2",
            is_moderator=False,
            moderator_on=None,
            columns=["control2"]
        )
    ]
)

cvars_text = _format_cvars_for_prompt(complex_cvars)

# Check all variables are present
assert "iv1" in cvars_text and "iv2" in cvars_text, "All IVs should be present"
assert "control1" in cvars_text and "control2" in cvars_text, "All controls should be present"
assert "dv" in cvars_text, "DV should be present"

# Check structure
lines = cvars_text.split("\n")
iv_section = [l for l in lines if "IV" in l or "iv1" in l or "iv2" in l]
assert len([l for l in iv_section if "iv1" in l or "iv2" in l]) >= 2, "Should list both IVs"

print("Test 6 passed: Multiple variables formatted correctly")
print(f"\nFound {len(complex_cvars.ivs)} IVs, {len(complex_cvars.controls)} controls")


Test 6 passed: Multiple variables formatted correctly

Found 2 IVs, 2 controls


## Test 7: Extract code with longer, more complex code


In [41]:
# Test with longer, more complex code
longer_transform = """def transform(df: pd.DataFrame) -> pd.DataFrame:
    # Make a copy to avoid modifying original
    df = df.copy()
    
    # Drop rows with missing values
    df = df.dropna(subset=['x', 'y'])
    
    # Create derived column
    df['derived'] = df['x'] * 2 + df['y']
    
    # Standardize
    df['x_z'] = (df['x'] - df['x'].mean()) / df['x'].std()
    
    return df"""

longer_model = """def model(df: pd.DataFrame) -> Any:
    import statsmodels.api as sm
    
    # Prepare data
    X = df[['x_z', 'derived']]
    X = sm.add_constant(X)
    y = df['y']
    
    # Fit model
    model = sm.OLS(y, X).fit()
    return model"""

longer_py_content = SAVE_CODE_TEMPLATE.format(
    data_path="/path/to/data.csv",
    transform_code=longer_transform,
    model_code=longer_model
)

with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(longer_py_content)
    temp_path = f.name

try:
    transform_code, model_code = _extract_code_from_file(temp_path)
    
    # Verify extraction preserved the code correctly
    assert "df.copy()" in transform_code, "Should preserve transform code"
    assert "sm.OLS" in model_code, "Should preserve model code"
    assert len(transform_code) > 100, "Should handle longer code"
    assert len(model_code) > 50, "Should handle longer code"
    
    print("Test 7 passed: Code extraction works with longer and more complex code")
    print(f"  Transform code: {len(transform_code)} chars")
    print(f"  Model code: {len(model_code)} chars")
finally:
    os.unlink(temp_path)


Test 7 passed: Code extraction works with longer and more complex code
  Transform code: 368 chars
  Model code: 235 chars


## Test 8: Integration test - full code fixing flow (with LLM calls)


In [43]:
# Set to True to run integration test with real LLM
# Set to False to skip (useful if credentials aren't available)
RUN_INTEGRATION_TEST = True

if RUN_INTEGRATION_TEST:
    import yaml
    from blade_bench.utils import get_dataset_csv_path
    
    # Load LLM config (same as analysis.ipynb uses)
    # From tests/stat_genie/blade_pipeline/baselines/, go up 4 levels to root, then into config/
    try:
        llm_config = yaml.safe_load(open("../../../../config/llm_eval_config.yml"))
        llm_provider = llm_config.get("provider", "openai")
        llm_model = llm_config.get("model", "gpt-5-mini")  # Same model as analysis.ipynb
        
        # Use a small dataset for testing (if available)
        test_dataset = "toy"  # or another small dataset
        try:
            dataset_path = get_dataset_csv_path(test_dataset)
        except:
            print("Test dataset not available, skipping integration test")
            RUN_INTEGRATION_TEST = False
    except Exception as e:
        print(f"Could not load LLM config: {e}")
        print("Skipping integration test")
        RUN_INTEGRATION_TEST = False

if RUN_INTEGRATION_TEST:
    print(f"LLM config loaded: {llm_provider}/{llm_model}")
    print(f"Dataset path: {dataset_path}")
else:
    print("Integration test skipped (set RUN_INTEGRATION_TEST = True to enable)")


LLM config loaded: openai/gpt-5-mini
Dataset path: /accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/blade_bench/datasets/toy/data.csv


In [44]:
if RUN_INTEGRATION_TEST:
    # Create a .py file with intentionally broken code
    # (missing import, wrong column name, etc.)
    broken_transform = """def transform(df: pd.DataFrame) -> pd.DataFrame:
    # This code has issues:
    # 1. Uses wrong column name 'x' instead of 'test_iv'
    # 2. Missing return statement initially (we'll make it syntactically valid but logically wrong)
    df = df.copy()
    df['test_iv'] = df['x']  # Wrong: 'x' doesn't exist, should use an existing column
    return df"""
    
    broken_model = """def model(df: pd.DataFrame) -> Any:
    # This should work once transform is fixed
    return df['test_iv'].mean()"""
    
    broken_py_content = SAVE_CODE_TEMPLATE.format(
        data_path=dataset_path,
        transform_code=broken_transform,
        model_code=broken_model
    )
    
    # Create temp file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(broken_py_content)
        temp_code_path = f.name
    
    try:
        # Format cvars for prompt
        cvars_text = _format_cvars_for_prompt(sample_cvars)
        
        print("Testing full code fixing flow...")
        print(f"   Code file: {temp_code_path}")
        print(f"   Dataset: {dataset_path}")
        print(f"   LLM: {llm_provider}/{llm_model}")
        print("\n   Note: You will see error messages below.")
        print("   The test intentionally creates broken code, detects the error,")
        print("   sends it to the LLM for fixing, then verifies the fix worked.\n")
        
        # Try to fix the code (uses LLM call)
        # verbose=True shows the error detection, which is expected
        iterations = check_and_fix_code_with_cvars(
            code_name="test_integration",
            code_path=temp_code_path,
            cvars_text=cvars_text,
            llm_provider=llm_provider,
            llm_model=llm_model,
            dataset_path=dataset_path,
            verbose=True
        )
        
        print("\n   Error detection and LLM fixing complete.\n")
        
        if iterations >= 0:
            print(f"Code fixed in {iterations} iteration(s)")
            
            # Extract the fixed code
            fixed_transform, fixed_model = _extract_code_from_file(temp_code_path)
            
            # Verify the fixed code is different (was actually fixed)
            assert fixed_transform != broken_transform, "Code should have been fixed"
            
            # Update analysis with fixed code
            updated_analysis = _update_analysis_with_fixed_code(
                sample_entire_analysis,
                fixed_transform,
                fixed_model
            )
            
            # Verify cvars preserved
            assert updated_analysis.cvars.model_dump() == sample_entire_analysis.cvars.model_dump()
            
            # Verify code updated
            assert updated_analysis.transform_code == fixed_transform
            assert updated_analysis.m_code == fixed_model
            
            print("Test 8 passed: Full integration flow works")
            print(f"   Code fixed: true")
            print(f"   Code extracted: true")
            print(f"   Analysis updated: true")
            print(f"   cvars preserved: true")
        else:
            print("Code fixing hit max iterations, but flow still works")
            print("Test 8 partially passed: Integration flow works (code may need manual fixing)")
    
    except Exception as e:
        print(f"Integration test failed: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if os.path.exists(temp_code_path):
            os.unlink(temp_code_path)
else:
    print("Integration test skipped")


Testing full code fixing flow...
   Code file: /tmp/tmp1w963ud_.py
   Dataset: /accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/blade_bench/datasets/toy/data.csv
   LLM: openai/gpt-5-mini

   Note: You will see error messages below.
   The test intentionally creates broken code, detects the error,
   sends it to the LLM for fixing, then verifies the fix worked.

Error during runtime:
  File "/accounts/campus/austin.zane/stat-genie/src/stat_genie/blade_pipeline/additions/analysis/fix_code.py", line 90, in is_code_correct
    transformed_df = transform_func(df)
                     ^^^^^^^^^^^^^^^^^^
  File "/tmp/tmp1w963ud_.py", line 19, in transform
    df['test_iv'] = df['x']  # Wrong: 'x' doesn't exist, should use an existing column
                    ~~^^^^^
  File "/accounts/campus/austin.zane/stat-genie/.venv/lib/python3.11/site-packages/pandas/core/frame.py", line 3807, in __getitem__
    indexer = self.columns.get_loc(key)
              ^^^^^^^^^^^^^^^